# Most of Your AI Bill Is Paying a Genius to Do a Clerk's Job

A logistics company gets **10,000 customer emails a day**. About 9 in 10 are simple; about 1 in 10 are hard (damage claims, disputes).

We compare three strategies: the **big model** for everything, the **small model** for everything, and a **router** that sends easy emails to the small model and hard ones to the big one.

The prices are examples and the skill levels are assumptions: change them to match your own provider and measurements. Run the cell below.

In [ ]:
# Lesson 05 — Model routing: how much of your AI bill pays a genius to do a clerk's job?
# 10,000 customer emails a day at a logistics company. Standard library only.
import random
random.seed(5)
HARD_SHARE = 0.10   # 1 in 10 emails is hard (damage claims, disputes); the rest are simple
# Example prices in $ per million tokens (input, output). Swap in your provider's real prices.
PRICE = {"big": (3.00, 15.00), "small": (0.15, 0.60)}
TOKENS_IN, TOKENS_OUT = 600, 150      # a typical email in, a short answer out
SKILL = {  # chance of handling an email correctly (assumed; measure your own)
    "big":   {"easy": 0.98, "hard": 0.92},
    "small": {"easy": 0.97, "hard": 0.55},
}
ROUTER_CATCHES_HARD = 0.90  # router spots 9 in 10 hard emails...
ROUTER_FALSE_ALARM = 0.05   # ...and wrongly escalates 1 in 20 easy ones

def cost(model):
    p_in, p_out = PRICE[model]
    return (TOKENS_IN * p_in + TOKENS_OUT * p_out) / 1e6

emails = ["hard" if random.random() < HARD_SHARE else "easy" for _ in range(10_000)]

def run(strategy):
    spend, wrong, wrong_hard = 0.0, 0, 0
    for kind in emails:
        if strategy == "router":
            spend += cost("small")  # the small model also does the sorting
            flagged = random.random() < (ROUTER_CATCHES_HARD if kind == "hard" else ROUTER_FALSE_ALARM)
            model = "big" if flagged else "small"
        else:
            model = strategy
        spend += cost(model)
        if random.random() > SKILL[model][kind]:
            wrong += 1
            wrong_hard += kind == "hard"
    return spend * 30, wrong, wrong_hard

print(f"{'strategy':10}{'$/month':>10}{'wrong/day':>11}{'of which hard':>15}")
for s in ["big", "small", "router"]:
    monthly, wrong, wrong_hard = run(s)
    print(f"{s:10}{monthly:>10,.0f}{wrong:>11}{wrong_hard:>15}")


## 5-minute tweak challenge

1. Set `ROUTER_CATCHES_HARD = 0.98`. How close does the router get to the big model on hard mistakes?
2. Set `HARD_SHARE = 0.40`. Is routing still worth building?
3. Halve the big model's prices. At what price does routing stop making sense for you?